# Phân loại Tin tức Kinh tế – Tài chính Tiếng Việt (VnEconomy) – Khám phá Dữ liệu Văn bản (Text EDA)

**Môn học:** Nền tảng lập trình cho Phân tích và Trực quan hóa Dữ liệu (CO5177)  
**Học kỳ:** HK261 (Năm học 2026 – 2027)  
**Giảng viên hướng dẫn:** TS. Lê Thành Sách  
**Nhóm:** G6  
**SVTH:** 
- Lê Đình Đức - 2310774
- Nguyễn Văn Công Thành - 2313133

---

## Giới thiệu

Notebook này thực hiện **EDA dữ liệu văn bản** và **chuẩn hóa tiền xử lý ngôn ngữ học** cho tập tin tức VnEconomy gồm **2.500 bài viết** thuộc **5 chuyên mục kinh tế – tài chính**, làm cơ sở cho bài toán phân loại chuyên mục ở notebook mô hình hóa.

**Nguyên tắc của notebook:** mọi con số trong phần *Nhận xét* đều lấy từ output của cell ngay phía trên; các phân tích có dùng nhãn (từ khóa đặc trưng, tâm chuyên mục, baseline) chỉ chạy trên tập **Train** (hoặc Train+Val), **tập Test được cố định ngay từ Mục 7 và không bị chạm tới**.

**Nội dung:**
1. Khai báo thư viện và cấu hình
2. Nạp dữ liệu và kiểm tra cấu trúc
3. Chất lượng dữ liệu và chuẩn hóa văn bản
4. Phân bố nhãn và xu hướng xuất bản
5. Độ dài văn bản, ngoại lai và cấu trúc câu
6. Bài gần trùng (near-duplicate) và chuỗi bài định kỳ
7. Chia tập có bảo toàn nhóm (Train / Val / Test)
8. Tách từ (`pyvi`), stopwords và độ phong phú từ vựng
9. Thống kê từ vựng và từ khóa đặc trưng chuyên mục
10. Cụm từ (N-gram)
11. Thực thể số và ký hiệu tài chính
12. Kiểm toán thuộc tính tác giả
13. Độ tách biệt giữa các chuyên mục và độ nhiễu nhãn
14. Tổng kết và kế hoạch mô hình hóa

## 1. Khai báo thư viện và cấu hình trực quan hóa

Yêu cầu môi trường: `seaborn>=0.13`, `scikit-learn>=1.3`, `pandas>=2.0`, `pyvi`. Nếu thiếu `pyvi`, cell dưới sẽ tự cài.

In [ ]:
import hashlib, pickle, re, subprocess, sys, unicodedata, warnings
from collections import Counter
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (adjusted_mutual_info_score, classification_report,
                             confusion_matrix, f1_score, normalized_mutual_info_score)
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer

try:
    from pyvi import ViTokenizer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyvi"])
    from pyvi import ViTokenizer

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({'figure.autolayout': True, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
                     'axes.labelsize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9})
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 160)

CATS = ['Bất động sản', 'Chứng khoán', 'Thế giới', 'Thị trường', 'Tài chính']
CAT_PALETTE = {'Bất động sản': '#E65100', 'Chứng khoán': '#1E88E5', 'Thế giới': '#43A047',
               'Thị trường': '#8E24AA', 'Tài chính': '#FDD835'}
TEXT_COLS = ['title', 'summary', 'content']

## 2. Nạp dữ liệu và kiểm tra cấu trúc

### 2.1 Nạp dữ liệu

Tập dữ liệu crawl từ VnEconomy nằm trong repo của nhóm: [`dducsw/co5177-data-analytics-and-visualization-project`](https://github.com/dducsw/co5177-data-analytics-and-visualization-project) (`text/crawler/data/vneconomy_articles.csv`). Cell dưới ưu tiên file cục bộ; nếu không có (ví dụ trên Colab) sẽ đọc trực tiếp từ GitHub.

In [ ]:
CSV_URL = ("https://raw.githubusercontent.com/dducsw/co5177-data-analytics-and-visualization-project/"
           "main/text/crawler/data/vneconomy_articles.csv")
local_candidates = [Path("text/crawler/data/vneconomy_articles.csv"),
                    Path("crawler/data/vneconomy_articles.csv"),
                    Path("../text/crawler/data/vneconomy_articles.csv"),
                    Path("vneconomy_articles.csv")]
csv_path = next((p for p in local_candidates if p.exists()), None)
print("Nguồn dữ liệu:", f"file cục bộ ({csv_path.name})" if csv_path else "GitHub (raw URL)")

df_raw = pd.read_csv(csv_path if csv_path else CSV_URL)   # bản gốc, không sửa
df = df_raw.copy()                                          # bản làm việc

print(f"- Số bài viết: {df.shape[0]:,} | Số cột: {df.shape[1]} | Bộ nhớ: {df.memory_usage(deep=True).sum() / 2**20:.1f} MB")
print(f"- Số dòng trùng hoàn toàn: {df.duplicated().sum()}")
print("- Các cột:", list(df.columns))

### 2.2 Ý nghĩa các cột

| Cột | Ý nghĩa | Vai trò trong bài toán |
| :--- | :--- | :--- |
| `id` | Mã băm MD5 của URL | Định danh duy nhất |
| `url` | Đường dẫn gốc | Truy vết nguồn |
| `title` | Tiêu đề | Văn bản ngắn, tóm lược cao |
| `summary` | Sapo / tóm tắt | Văn bản trung bình |
| `content` | Thân bài | Văn bản dài chính |
| `word_count` | Số "từ" do crawler đếm theo khoảng trắng (trước chuẩn hóa) | Tham chiếu độ dài |
| `raw_category` | Chuyên mục gốc trên website | Nhãn gốc |
| `category` | Chuyên mục mục tiêu (5 lớp) | **Nhãn phân loại** |
| `published_date` | Ngày giờ xuất bản | Khảo sát thời gian |
| `published_year_month` | Tháng-năm xuất bản (dẫn xuất) | Khảo sát thời gian |
| `author` | Tác giả / phóng viên | Kiểm toán rò rỉ |

In [ ]:
display(df[['title', 'category', 'raw_category', 'word_count', 'published_date', 'author']].head(3))
print("raw_category == category trên toàn bộ dữ liệu:", bool((df['raw_category'] == df['category']).all()))

# Mẫu ngẫu nhiên (có seed) mỗi chuyên mục - không chọn bài đầu tiên để tránh thiên lệch
print("\n" + "=" * 90)
for cat in CATS:
    r = df[df['category'] == cat].sample(1, random_state=SEED).iloc[0]
    print(f"[{cat.upper()}] {r['title']}")
    print(f"   Sapo: {r['summary'][:200]}...")

**Nhận xét:** @@sec2@@

## 3. Chất lượng dữ liệu và chuẩn hóa văn bản

### 3.1 Dữ liệu thiếu và trùng lặp

In [ ]:
na_counts = df.isna().sum()
blank_counts = {c: int(df[c].astype(str).str.strip().eq("").sum()) for c in df.columns}
quality_df = pd.DataFrame({'Kiểu dữ liệu': df.dtypes.astype(str), 'NaN': na_counts,
                           'Chuỗi rỗng': pd.Series(blank_counts)})
display(quality_df)
print(f"Trùng lặp: id={df.duplicated('id').sum()}, url={df.duplicated('url').sum()}, content={df.duplicated('content').sum()}")
print("\nBản ghi thiếu ngày xuất bản:")
display(df.loc[df['published_date'].isna(), ['title', 'category', 'published_date']])

**Nhận xét:** @@sec3_1@@

### 3.2 Chuẩn hóa Unicode, ký tự ẩn, URL và vị trí dấu thanh

Văn bản báo điện tử thường lẫn Unicode dựng sẵn (NFC) với tổ hợp (NFD), và hai cách đặt dấu thanh cho vần *oa, oe, uy* (*hòa* / *hoà*, *thủy* / *thuỷ*). Nếu không thống nhất, TF-IDF sẽ coi chúng là hai từ khác nhau. Nhóm chuẩn hóa:
1. Đưa về Unicode NFC.
2. Thống nhất vị trí dấu thanh theo kiểu truyền thống (*hoà, khoẻ, thuỷ*) cho vần **oa, oe, uy**.
3. Xóa URL, ký tự ẩn (zero-width, NBSP) và khoảng trắng thừa, **giữ nguyên ngắt dòng** để đếm câu/tiêu đề phụ ở Mục 5.

> **Lưu ý:** không chuẩn hóa vần *ua* (*của, mùa, lúa*) vì dấu thanh luôn nằm trên *u*; chuẩn hóa nhầm sẽ biến *của → cuả* và làm hỏng stopword.

In [ ]:
# Kiểu mới (òa, ủy...) -> kiểu truyền thống (oà, uỷ...). KHÔNG đụng tới vần 'ua'.
TONE_MAP = {
    'òa': 'oà', 'óa': 'oá', 'ỏa': 'oả', 'õa': 'oã', 'ọa': 'oạ',
    'òe': 'oè', 'óe': 'oé', 'ỏe': 'oẻ', 'õe': 'oẽ', 'ọe': 'oẹ',
    'ùy': 'uỳ', 'úy': 'uý', 'ủy': 'uỷ', 'ũy': 'uỹ', 'ụy': 'uỵ'}
RE_TONE = re.compile("|".join(re.escape(k) for k in TONE_MAP))
URL_RE = re.compile(r"https?://\S+|www\.\S+")
HIDDEN_CHARS_RE = re.compile("[\u200b-\u200d\ufeff\u00a0]")

def normalize_text_vietnamese(text):
    if not isinstance(text, str):
        return ""
    t = unicodedata.normalize("NFC", text)
    t = HIDDEN_CHARS_RE.sub(" ", t)
    t = RE_TONE.sub(lambda m: TONE_MAP[m.group(0)], t)
    t = URL_RE.sub(" ", t)
    t = re.sub(r"[^\S\r\n]+", " ", t)          # gộp khoảng trắng, GIỮ ngắt dòng
    t = re.sub(r"\n\s*\n+", "\n\n", t)
    return t.strip()

# Đo trước/sau: số bài chưa NFC, và số lần xuất hiện hai kiểu đặt dấu (loại trừ 'qu' như quý, quỹ, quỳ)
NEW_STYLE = re.compile("(?<![qQ])(?:" + "|".join(k for k in TONE_MAP) + ")")
OLD_STYLE = re.compile("(?<![qQ])(?:" + "|".join(TONE_MAP.values()) + ")")
def count_styles(frame):
    blob = " ".join(frame[c].astype(str).str.cat(sep=" ") for c in TEXT_COLS)
    return len(NEW_STYLE.findall(blob)), len(OLD_STYLE.findall(blob))
def n_non_nfc(frame):
    return int(frame[TEXT_COLS].apply(lambda s: s.map(lambda x: not unicodedata.is_normalized("NFC", str(x)))).any(axis=1).sum())

new_b, old_b = count_styles(df); nfc_b = n_non_nfc(df)
for col in TEXT_COLS:
    df[col] = df[col].map(normalize_text_vietnamese)
new_a, old_a = count_styles(df); nfc_a = n_non_nfc(df)

display(pd.DataFrame({'Trước': [nfc_b, new_b, old_b], 'Sau': [nfc_a, new_a, old_a]},
                     index=['Số bài chưa chuẩn NFC', 'Số lần dấu kiểu mới (hòa, thủy)', 'Số lần dấu kiểu truyền thống (hoà, thuỷ)']))

# Kiểm tra an toàn: các từ có vần 'ua' không bị biến dạng
probe = normalize_text_vietnamese("Giá lúa mùa của chúa; hòa bình, thủy sản, khỏe")
print("Kiểm tra:", probe)

**Nhận xét:** @@sec3_2@@

## 4. Phân bố nhãn và xu hướng xuất bản

### 4.1 Phân bố 5 chuyên mục và theo tháng

In [ ]:
label_counts = df['category'].value_counts().reindex(CATS)
display(label_counts.rename('Số bài').to_frame().assign(**{'Tỷ lệ (%)': lambda x: (x['Số bài'] / len(df) * 100).round(1)}).T)

df['dt'] = pd.to_datetime(df['published_date'], errors='coerce')
dt_valid = df.loc[df['dt'].notna()].copy()
print(f"Khoảng thời gian: {dt_valid['dt'].min():%d/%m/%Y} → {dt_valid['dt'].max():%d/%m/%Y} ({(dt_valid['dt'].max() - dt_valid['dt'].min()).days} ngày)")
month_cat = pd.crosstab(dt_valid['dt'].dt.to_period('M').astype(str), dt_valid['category'])[CATS]
display(month_cat.T)

**Nhận xét:** @@sec4_1@@

### 4.2 Nhịp đăng bài trong tuần và khung giờ

Số bài theo **tháng** bị quota của crawler khống chế (Mục 4.1), nhưng nhịp theo **thứ trong tuần** và **giờ trong ngày** vẫn phản ánh thói quen đăng bài của tòa soạn.

In [ ]:
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
days_vn = ['Thứ 2', 'Thứ 3', 'Thứ 4', 'Thứ 5', 'Thứ 6', 'Thứ 7', 'Chủ nhật']
dow = dt_valid['dt'].dt.day_name().map(dict(zip(days_order, days_vn))).value_counts().reindex(days_vn)
hour = dt_valid['dt'].dt.hour.value_counts().reindex(range(24), fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
axes[0].bar(dow.index, dow.values, color='#42A5F5'); axes[0].set_title("Số bài theo thứ trong tuần"); axes[0].set_ylabel("Số bài")
axes[1].bar(hour.index, hour.values, color='#66BB6A'); axes[1].set_title("Số bài theo giờ trong ngày"); axes[1].set_xlabel("Giờ"); axes[1].set_xticks(range(0, 24, 2))
plt.show()

print("Số bài theo thứ:", dow.to_dict())
print(f"Tỷ lệ bài đăng Thứ 2–Thứ 6: {dow[days_vn[:5]].sum() / dow.sum() * 100:.1f}% | Thứ 7+CN: {dow[days_vn[5:]].sum() / dow.sum() * 100:.1f}%")
print("Top 5 giờ đăng nhiều nhất:", {int(h): int(v) for h, v in hour.sort_values(ascending=False).head(5).items()})
print(f"Tỷ lệ bài đăng 8h–16h: {hour.loc[8:16].sum() / hour.sum() * 100:.1f}%")

**Nhận xét:** @@sec4_2@@

## 5. Độ dài văn bản, ngoại lai và cấu trúc câu

### 5.1 Độ dài theo âm tiết

Số âm tiết = số token tách theo khoảng trắng (tiếng Việt viết rời từng âm tiết). Số ký tự tương quan gần như tuyệt đối với số âm tiết nên chỉ dùng âm tiết.

In [ ]:
for col in TEXT_COLS:
    df[f"{col}_syl"] = df[col].map(lambda x: len(str(x).split()))

len_summary = df[[f"{c}_syl" for c in TEXT_COLS]].describe(percentiles=[.25, .5, .75, .95]).T
len_summary.index = ['Tiêu đề', 'Sapo', 'Thân bài']
display(len_summary[['mean', 'std', 'min', '25%', '50%', '75%', '95%', 'max']].round(1))
print(f"Tương quan Spearman số ký tự ~ số âm tiết (thân bài): {df['content'].str.len().corr(df['content_syl'], method='spearman'):.3f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
sns.histplot(df['title_syl'], ax=axes[0], bins=25, kde=True, color='#1E88E5'); axes[0].set_title("Độ dài tiêu đề (âm tiết)")
sns.histplot(df['summary_syl'], ax=axes[1], bins=25, kde=True, color='#43A047'); axes[1].set_title("Độ dài sapo (âm tiết)")
sns.boxplot(data=df, x='category', y='content_syl', order=CATS, hue='category', palette=CAT_PALETTE, legend=False, ax=axes[2])
axes[2].set_title("Độ dài thân bài theo chuyên mục"); axes[2].set_xlabel(""); axes[2].set_ylabel("Âm tiết"); axes[2].tick_params(axis='x', rotation=20)
plt.show()

print("Trung vị độ dài thân bài (âm tiết) theo chuyên mục:")
print(df.groupby('category')['content_syl'].median().reindex(CATS).round(0).astype(int).to_dict())
print(f"\nĐối soát `word_count` của crawler với số âm tiết sau chuẩn hóa: Spearman = {df['word_count'].corr(df['content_syl'], method='spearman'):.3f}; "
      f"tỷ lệ trung vị word_count / content_syl = {(df['word_count'] / df['content_syl']).median():.2f}")

**Nhận xét:** @@sec5_1@@

### 5.2 Ngoại lai độ dài (quy tắc IQR)

In [ ]:
q1, q3 = df['content_syl'].quantile([.25, .75])
iqr = q3 - q1
upper, lower = q3 + 1.5 * iqr, q1 - 1.5 * iqr
n_hi, n_lo = int((df['content_syl'] > upper).sum()), int((df['content_syl'] < lower).sum())
print(f"Ngưỡng IQR (âm tiết): dưới = {lower:.0f} | trên = {upper:.0f}")
print(f"Ngoại lai trên: {n_hi} bài ({n_hi / len(df) * 100:.1f}%) | ngoại lai dưới: {n_lo} bài | bài ngắn nhất: {df['content_syl'].min()} âm tiết")
print("Phân bố ngoại lai trên theo chuyên mục:", df.loc[df['content_syl'] > upper, 'category'].value_counts().to_dict())

**Nhận xét:** @@sec5_2@@

### 5.3 Quan hệ sapo – thân bài và cấu trúc câu

In [ ]:
def summary_in_body(row, k=50):
    s = str(row['summary'])[:k].lower()
    return (s in str(row['content']).lower()) if len(s) >= 20 else False
df['summary_in_body'] = df.apply(summary_in_body, axis=1)

PROTECT_ABBR = re.compile(r"\b(TP|TS|PGS|GS|ThS|BS|KS|ĐH|Tp|TT|Q|P)\.(?=\s+[A-ZÀ-Ỹ])")
def count_sentences_and_heads(text):
    sents = heads = 0
    for line in text.split('\n'):
        line = line.strip()
        if not line:
            continue
        if len(line.split()) <= 15 and not line.endswith(('.', '!', '?')) and line.isupper():
            heads += 1
            continue
        s = re.sub(r"(?<=\d)\.(?=\d)", "<DOT>", line)
        s = PROTECT_ABBR.sub(r"\1<DOT>", s)
        sents += len([p for p in re.split(r"[\.\!\?]+", s) if p.strip()])
    return max(1, sents), heads

res = df['content'].map(count_sentences_and_heads)
df['n_sent'] = [r[0] for r in res]; df['n_heads'] = [r[1] for r in res]
df['syl_per_sent'] = df['content_syl'] / df['n_sent']

print(f"Sapo lặp nguyên văn ở thân bài: {df['summary_in_body'].mean() * 100:.1f}% số bài")
print(f"Số câu/bài: TB {df['n_sent'].mean():.1f} (trung vị {df['n_sent'].median():.0f}) | âm tiết/câu: TB {df['syl_per_sent'].mean():.1f}")
print(f"Tiêu đề phụ (dòng IN HOA ngắn): {int(df['n_heads'].sum())} dòng trong {int((df['n_heads'] > 0).sum())} bài")

**Nhận xét:** @@sec5_3@@

## 6. Bài gần trùng (near-duplicate) và chuỗi bài định kỳ

Các bài cùng khuôn mẫu (bản tin phiên, bình luận định kỳ) nếu rơi vào cả Train lẫn Test sẽ gây **rò rỉ dữ liệu**. Nhóm kiểm tra hai tín hiệu: (i) độ tương đồng cosine giữa các bài (TF-IDF ký tự 4–5-gram), (ii) **tiền tố tiêu đề lặp** (ví dụ `Blog chứng khoán:`).

### 6.1 Độ tương đồng cosine giữa các bài

In [ ]:
norm_text = df['content'].str.lower().str.replace(r"\s+", " ", regex=True)
vec_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(4, 5), min_df=2, max_features=50000, sublinear_tf=True)
Xc = vec_char.fit_transform(norm_text)
sim = (Xc @ Xc.T).toarray()
np.fill_diagonal(sim, 0)

iu = np.triu_indices_from(sim, 1)
sim_vals = sim[iu]
print(f"Cosine giữa hai bài bất kỳ: TB = {sim_vals.mean():.3f} | p99 = {np.percentile(sim_vals, 99):.3f} | max = {sim_vals.max():.3f}")

rows = []
for th in [0.70, 0.75, 0.80, 0.85, 0.90]:
    _, lab = connected_components(csr_matrix(sim >= th), directed=False)
    sz = np.bincount(lab); multi = sz[sz > 1]
    rows.append({'Ngưỡng cosine': th, 'Số nhóm (>1 bài)': len(multi), 'Số bài thuộc nhóm': int(multi.sum()),
                 'Nhóm lớn nhất': int(multi.max()) if len(multi) else 0})
display(pd.DataFrame(rows))

top = np.argsort(sim_vals)[::-1][:6]
pairs = pd.DataFrame({'cosine': sim_vals[top].round(3),
                      'bài A': df['title'].values[iu[0][top]], 'bài B': df['title'].values[iu[1][top]],
                      'chuyên mục': df['category'].values[iu[0][top]]})
print("Các cặp bài giống nhau nhất:")
display(pairs)

**Nhận xét:** @@sec6_1@@

### 6.2 Chuỗi bài định kỳ qua tiền tố tiêu đề

Với mỗi tiền tố xuất hiện ≥ 5 bài, nhóm so sánh **độ tương đồng trung bình giữa các bài trong chuỗi** với mức nền toàn tập để quyết định chuỗi nào cần được gom nhóm.

In [ ]:
prefix = df['title'].str.extract(r"^([^:]{3,30}):")[0]
cand = prefix.value_counts()
cand = cand[cand >= 5]
rows = []
for p, n in cand.items():
    idx = np.where(prefix == p)[0]
    sub = sim[np.ix_(idx, idx)]
    rows.append({'Tiền tố': p, 'Số bài': n, 'Chuyên mục': df.loc[idx, 'category'].value_counts().to_dict(),
                 'Cosine TB trong chuỗi': sub[np.triu_indices(len(idx), 1)].mean().round(3)})
series_df = pd.DataFrame(rows)
display(series_df)
print(f"Mức nền (cosine TB giữa hai bài bất kỳ): {sim_vals.mean():.3f}")

**Nhận xét:** @@sec6_2@@

### 6.3 Gán nhóm `dup_group`

Hai bài thuộc cùng một nhóm nếu **cosine ≥ 0,85** *hoặc* cùng một **chuỗi định kỳ** (tiền tố có cosine trong chuỗi cao hơn rõ rệt mức nền, ngưỡng `SERIES_MIN_SIM`). Các nhóm là thành phần liên thông của đồ thị này.

In [ ]:
SERIES_MIN_SIM = 0.35        # chuỗi định kỳ: cosine TB trong chuỗi >= ngưỡng này (nền ~ 0.1)
series_prefixes = series_df.loc[series_df['Cosine TB trong chuỗi'] >= SERIES_MIN_SIM, 'Tiền tố'].tolist()
print("Chuỗi định kỳ được gom nhóm:", series_prefixes)
print("Chuỗi bị loại (tiền tố trùng nhưng nội dung khác nhau):", [p for p in cand.index if p not in series_prefixes])

adj = sim >= 0.85
for p in series_prefixes:
    idx = np.where(prefix == p)[0]
    adj[np.ix_(idx, idx)] = True
_, df['dup_group'] = connected_components(csr_matrix(adj), directed=False)

gsz = df['dup_group'].value_counts()
multi_groups = gsz[gsz > 1]
print(f"\nSố nhóm > 1 bài: {len(multi_groups)} | tổng số bài thuộc các nhóm này: {int(multi_groups.sum())} | nhóm lớn nhất: {int(multi_groups.max())} bài")
for g in multi_groups.index[:6]:
    sub = df[df['dup_group'] == g]
    print(f"  nhóm {g}: {len(sub)} bài, {sub['category'].value_counts().to_dict()}, ví dụ: {sub['title'].iloc[0][:60]}")

**Nhận xét:** @@sec6_3@@

## 7. Chia tập có bảo toàn nhóm (Train / Val / Test)

Dùng **`StratifiedGroupKFold`**: *phân tầng* để giữ tỷ lệ 5 chuyên mục, *bảo toàn nhóm* để mọi bài cùng `dup_group` nằm trong cùng một tập. Chia 7-fold lấy 1 fold làm **Test (~14,3%)**; trong phần còn lại chia 6-fold lấy 1 fold làm **Val (~14,3%)**, còn lại là **Train (~71,4%)**.

Từ Mục 8 trở đi, các phân tích dùng nhãn (từ khóa, tâm chuyên mục, baseline) **chỉ dùng Train hoặc Train+Val**; Test không được chạm tới.

In [ ]:
y_all, groups = df['category'].values, df['dup_group'].values

sg1 = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
dev_idx, test_idx = next(sg1.split(df, y_all, groups))
sg2 = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
tr_rel, va_rel = next(sg2.split(dev_idx, y_all[dev_idx], groups[dev_idx]))
train_idx, val_idx = dev_idx[tr_rel], dev_idx[va_rel]

df['split'] = 'train'
df.iloc[val_idx, df.columns.get_loc('split')] = 'val'
df.iloc[test_idx, df.columns.get_loc('split')] = 'test'

split_tab = pd.crosstab(df['split'], df['category'])[CATS].loc[['train', 'val', 'test']]
split_tab['Tổng'] = split_tab.sum(axis=1)
split_tab['Tỷ lệ (%)'] = (split_tab['Tổng'] / len(df) * 100).round(1)
display(split_tab)

leak = {f"{a}-{b}": len(set(groups[ia]) & set(groups[ib]))
        for (a, ia), (b, ib) in [(('train', train_idx), ('val', val_idx)), (('train', train_idx), ('test', test_idx)), (('val', val_idx), ('test', test_idx))]}
print("Số nhóm bị rò rỉ giữa các tập (kỳ vọng = 0):", leak)

# Lưu để notebook mô hình hóa dùng đúng cùng một phép chia
df[['id', 'category', 'dup_group', 'split']].to_csv("eda_split.csv", index=False)
print("Đã lưu eda_split.csv")

TR, VA, TE = (df['split'] == 'train').values, (df['split'] == 'val').values, (df['split'] == 'test').values
DEV = TR | VA

**Nhận xét:** @@sec7@@

## 8. Tách từ (`pyvi`), stopwords và độ phong phú từ vựng

### 8.1 Tách từ bằng PyVi

Tách từ là bước không dùng nhãn nên chạy trên toàn bộ dữ liệu. Kết quả được **cache** ra file (kèm mã băm của văn bản) để không phải tách lại mỗi lần mở notebook.

In [ ]:
PUNCT_EXCLUDE = re.compile(r"^[\W\d_]+$")

def prep_for_tokenizer(text):
    t = URL_RE.sub(" ", str(text))
    t = re.sub(r"(?i)\b(vn|hnx|upcom)\s*-\s*index\b", lambda m: m.group(1).lower() + "index", t)
    t = re.sub(r"(?<=\d)[.,](?=\d)", "_", t)
    t = re.sub(r"([^\w\s])", r" \1 ", t)
    return re.sub(r"\s+", " ", t).strip()

def tokenize_segments(text):
    tokenized = ViTokenizer.tokenize(prep_for_tokenizer(text))
    return [w for w in tokenized.split() if not PUNCT_EXCLUDE.match(w)]

CACHE = Path("tokenized_cache.pkl")
text_hash = hashlib.md5("".join(df['title'] + df['summary'] + df['content']).encode()).hexdigest()
cache = pickle.load(open(CACHE, "rb")) if CACHE.exists() else {}
if cache.get('hash') == text_hash:
    print("Nạp kết quả tách từ từ cache.")
    tok_title, tok_summary, tok_content = cache['title'], cache['summary'], cache['content']
else:
    print("Tách từ 2.500 bài (lần đầu, có thể mất vài phút)...")
    tok_title = [tokenize_segments(t) for t in tqdm(df['title'], desc='title')]
    tok_summary = [tokenize_segments(t) for t in tqdm(df['summary'], desc='summary')]
    tok_content = [tokenize_segments(t) for t in tqdm(df['content'], desc='content')]
    pickle.dump({'hash': text_hash, 'title': tok_title, 'summary': tok_summary, 'content': tok_content}, open(CACHE, "wb"))

df['seg_doc'] = [a + b + c for a, b, c in zip(tok_title, tok_summary, tok_content)]
df['tok_doc_str'] = df['seg_doc'].map(" ".join)
df['n_words'] = df['seg_doc'].map(len)

sy_all = df['title_syl'] + df['summary_syl'] + df['content_syl']
print(f"Số token TB mỗi bài sau tách từ: {df['n_words'].mean():.1f} (so với {sy_all.mean():.1f} âm tiết của title+summary+content) "
      f"→ giảm {(1 - df['n_words'].mean() / sy_all.mean()) * 100:.1f}%")
print("Ví dụ:", " ".join(df['seg_doc'].iloc[0][:25]))

**Nhận xét:** @@sec8_1@@

### 8.2 Stopwords: loại hư từ, kiểm tra độ lệch theo chuyên mục

Danh sách gồm hư từ/từ chức năng và một số từ chỉ thời gian, số đếm phổ thông. Để chắc chắn việc loại bỏ không làm mất tín hiệu phân loại, nhóm tính **tỷ lệ bài chứa từ đó (DF) ở từng chuyên mục** và xem từ nào lệch nhất.

In [ ]:
STOPWORDS = {
    "và", "của", "các", "trong", "được", "cho", "những", "với", "về", "có", "là", "đã", "này", "đó", "khi", "ở", "từ",
    "đến", "theo", "nhiều", "sau", "trên", "dưới", "để", "như", "ra", "lại", "cũng", "một", "hai", "ba", "bốn", "sẽ", "bị",
    "do", "bởi", "tại", "qua", "nhưng", "mà", "nếu", "thì", "còn", "chỉ", "rất", "hơn", "hay", "hoặc", "vì", "vậy", "nên",
    "người", "việc", "ngày", "năm", "tháng", "tới", "trước", "cùng", "khác", "mới", "đang", "từng", "luôn", "gì", "nào",
    "ai", "đâu", "đây", "đấy", "thế", "nữa", "rồi", "vẫn", "chưa", "hết", "chính", "thực_hiện", "cho_biết", "cho_rằng"}

doc_sets = df['seg_doc'].map(set)
df_share = pd.DataFrame({c: [np.mean([w in s for s in doc_sets[df['category'] == c]]) for w in sorted(STOPWORDS)] for c in CATS},
                        index=sorted(STOPWORDS))
common = df_share[df_share.mean(axis=1) >= 0.05]          # chỉ xét stopword đủ phổ biến (DF TB >= 5%)
skew = (common.max(axis=1) / common.min(axis=1).clip(lower=1e-3)).sort_values(ascending=False)
print(f"Số stopwords: {len(STOPWORDS)} | số stopword xuất hiện trong >= 5% bài: {len(common)}")
print("Top 5 stopword phổ biến nhất (DF toàn tập):", (df_share.mean(axis=1).sort_values(ascending=False).head(5) * 100).round(1).to_dict())
display(pd.DataFrame({'Tỷ lệ lệch max/min DF': skew.head(5).round(2),
                      'Chuyên mục cao nhất': common.loc[skew.head(5).index].idxmax(axis=1),
                      'Chuyên mục thấp nhất': common.loc[skew.head(5).index].idxmin(axis=1)}))
all_tok = [w for d in df['seg_doc'] for w in d]
print(f"Stopwords chiếm {np.mean([w in STOPWORDS for w in all_tok]) * 100:.1f}% tổng số token")

**Nhận xét:** @@sec8_2@@

### 8.3 Độ phong phú từ vựng: TTR thô và MATTR

TTR thô phụ thuộc độ dài văn bản nên không so sánh được giữa các bài; **MATTR** (cửa sổ trượt 100 token) đã chuẩn hóa độ dài.

In [ ]:
def mattr(tokens, w=100):
    n = len(tokens)
    if n < w:
        return np.nan
    c = Counter(tokens[:w]); vals = [len(c) / w]
    for i in range(w, n):
        out = tokens[i - w]; c[out] -= 1
        if c[out] == 0:
            del c[out]
        c[tokens[i]] += 1
        vals.append(len(c) / w)
    return float(np.mean(vals))

df['ttr'] = df['seg_doc'].map(lambda t: len(set(t)) / len(t))
df['mattr'] = df['seg_doc'].map(mattr)

fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
sns.boxplot(data=df, x='category', y='mattr', order=CATS, hue='category', palette=CAT_PALETTE, legend=False, ax=axes[0])
axes[0].set_title("MATTR (cửa sổ 100 token) theo chuyên mục"); axes[0].set_xlabel(""); axes[0].tick_params(axis='x', rotation=20)
sns.scatterplot(data=df, x='n_words', y='ttr', hue='category', hue_order=CATS, palette=CAT_PALETTE, alpha=.5, s=14, ax=axes[1])
axes[1].set_title("TTR thô giảm theo độ dài bài"); axes[1].set_xlabel("Số token"); axes[1].set_ylabel("TTR thô")
plt.show()

print(f"Spearman(TTR thô, độ dài bài) = {df['ttr'].corr(df['n_words'], method='spearman'):.2f} | Spearman(MATTR, độ dài bài) = {df['mattr'].corr(df['n_words'], method='spearman'):.2f}")
display(df.groupby('category')[['mattr', 'ttr']].mean().reindex(CATS).round(3).rename(columns={'mattr': 'MATTR TB', 'ttr': 'TTR thô TB'}).T)

**Nhận xét:** @@sec8_3@@

## 9. Thống kê từ vựng và từ khóa đặc trưng chuyên mục

> Từ đây các phân tích có dùng nhãn chỉ chạy trên **tập Train**.

### 9.1 Quy mô từ vựng và tỷ lệ từ ngoài từ điển (OOV)

Thay vì kiểm chứng định luật Zipf/Heaps (đúng với hầu hết ngữ liệu và không dẫn tới quyết định nào), nhóm đo trực tiếp các đại lượng phục vụ chọn `min_df` / `max_features`: kích thước từ vựng, tỷ lệ từ xuất hiện đúng 1 lần (hapax) và **tỷ lệ OOV** của Val/Test so với từ vựng Train.

In [ ]:
train_docs = df.loc[TR, 'seg_doc']
cnt_train = Counter(w for d in train_docs for w in d)
n_tok_train, V_train = sum(cnt_train.values()), len(cnt_train)
hapax = sum(1 for c in cnt_train.values() if c == 1)

cv5 = CountVectorizer(token_pattern=r"\S+", lowercase=False, min_df=5).fit(df.loc[TR, 'tok_doc_str'])
vocab5 = set(cv5.get_feature_names_out())

def oov(mask, vocab):
    toks = [w for d in df.loc[mask, 'seg_doc'] for w in d]
    return np.mean([w not in vocab for w in toks]) * 100

display(pd.DataFrame({
    'Giá trị': [n_tok_train, V_train, f"{hapax:,} ({hapax / V_train * 100:.1f}% từ vựng)", len(vocab5),
                f"{oov(VA, set(cnt_train)):.2f}% / {oov(TE, set(cnt_train)):.2f}%",
                f"{oov(VA, vocab5):.2f}% / {oov(TE, vocab5):.2f}%"]},
    index=['Tổng token (Train)', 'Kích thước từ vựng (Train)', 'Hapax', 'Từ vựng sau min_df=5',
           'OOV Val / Test (so với toàn bộ từ vựng Train)', 'OOV Val / Test (so với từ vựng min_df=5)']))

**Nhận xét:** @@sec9_1@@

### 9.2 Top 20 từ nội dung phổ biến nhất (Train)

In [ ]:
top20 = Counter({w: c for w, c in cnt_train.items() if w not in STOPWORDS and len(w) > 1}).most_common(20)
w20, c20 = zip(*top20)
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.barh([w.replace('_', ' ') for w in w20][::-1], list(c20)[::-1], color='#3F7CAC')
for i, v in enumerate(list(c20)[::-1]):
    ax.text(v + max(c20) * 0.01, i, f"{v:,}", va='center', fontsize=8.5)
ax.set_title("Top 20 từ nội dung (sau khi loại stopword) - tập Train"); ax.set_xlabel("Số lần xuất hiện")
plt.show()
print("Top 20:", [w for w in w20])

**Nhận xét:** @@sec9_2@@

### 9.3 Từ khóa đặc trưng từng chuyên mục (Log-odds Z-score, Train)

Log-odds có prior thông tin (Monroe et al.) cho điểm Z để so sánh được giữa các từ. Từ nào lọt top 15 của **từ 2 chuyên mục trở lên** được đánh dấu `*` (dấu hiệu chồng lấn ngữ nghĩa giữa các lớp).

In [ ]:
cv = CountVectorizer(token_pattern=r"\S+", lowercase=False, stop_words=sorted(STOPWORDS), min_df=5)
Xcnt = cv.fit_transform(df.loc[TR, 'tok_doc_str'])
vocab = np.array(cv.get_feature_names_out())
y_tr = df.loc[TR, 'category'].values

def log_odds_z(X, mask, a0=500.0):
    total = np.asarray(X.sum(0)).ravel().astype(float); prior = total / total.sum()
    y_sum = np.asarray(X[mask].sum(0)).ravel().astype(float); n_y = y_sum.sum()
    bg_sum = total - y_sum; n_bg = bg_sum.sum()
    y_hat = (y_sum + a0 * prior) / (n_y + a0); bg_hat = (bg_sum + a0 * prior) / (n_bg + a0)
    beta = np.log(y_hat / (1 - y_hat)) - np.log(bg_hat / (1 - bg_hat))
    var = 1 / (y_sum + a0 * prior) + 1 / (bg_sum + a0 * prior)
    return beta / np.sqrt(var)

TOPK = 15
zs = {c: log_odds_z(Xcnt, y_tr == c) for c in CATS}
top_words = {c: vocab[np.argsort(zs[c])[::-1][:TOPK]] for c in CATS}
word_cnt = Counter(w for c in CATS for w in top_words[c])
shared = {w for w, n in word_cnt.items() if n >= 2}

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
for ax, c in zip(axes, CATS):
    idx = np.argsort(zs[c])[::-1][:TOPK]
    labels = [vocab[i].replace('_', ' ') + (' *' if vocab[i] in shared else '') for i in idx]
    ax.barh(labels[::-1], zs[c][idx][::-1], color=CAT_PALETTE[c], edgecolor='#444', linewidth=.4)
    ax.set_title(c); ax.set_xlabel("Log-odds Z")
plt.show()

top_keywords_dict = top_words
for c in CATS:
    print(f"{c:13s}: {', '.join(w for w in top_words[c][:10])}")
print("\nTừ nằm trong top 15 của >= 2 chuyên mục:", sorted(shared))

**Nhận xét:** @@sec9_3@@

### 9.4 Ma trận TF-IDF trung bình của từ khóa đặc trưng (Train)

Top 3 từ khóa log-odds của mỗi chuyên mục; vector hóa TF-IDF trên toàn từ vựng Train (không fit trên tập từ khóa nhỏ để tránh thổi phồng trọng số do chuẩn hóa L2), rồi lấy trung bình theo chuyên mục.

In [ ]:
sel_pairs = list(dict.fromkeys((w, c) for c in CATS for w in top_words[c][:3]))
selected = list(dict.fromkeys(w for w, _ in sel_pairs))
heat_tfidf = TfidfVectorizer(token_pattern=r"\S+", lowercase=False, stop_words=sorted(STOPWORDS), min_df=3, sublinear_tf=True)
X_heat = heat_tfidf.fit_transform(df.loc[TR, 'tok_doc_str'])
hv = {w: i for i, w in enumerate(heat_tfidf.get_feature_names_out())}
sel = [w for w in selected if w in hv]
cross = np.vstack([np.asarray(X_heat[y_tr == c][:, [hv[w] for w in sel]].mean(axis=0)).ravel() for c in CATS])
cross_df = pd.DataFrame(cross, index=CATS, columns=[w.replace('_', ' ') for w in sel])

plt.figure(figsize=(13, 4.4))
sns.heatmap(cross_df, annot=True, fmt='.3f', cmap='YlGnBu', linewidths=.6)
plt.title("TF-IDF trung bình của top-3 từ khóa mỗi chuyên mục (Train)"); plt.xticks(rotation=45, ha='right')
plt.show()

home = {w: c for w, c in sel_pairs}
rank_hit = np.mean([cross_df.iloc[:, k].idxmax() == home[w] for k, w in enumerate(sel)])
print(f"Tỷ lệ từ khóa có TF-IDF TB cao nhất đúng ở chuyên mục mà nó được chọn: {rank_hit * 100:.0f}% ({len(sel)} từ)")

**Nhận xét:** @@sec9_4@@

### 9.5 Chuỗi bài định kỳ có làm méo từ khóa không?

So sánh top-10 từ khóa log-odds của từng chuyên mục khi dùng toàn bộ Train với khi **chỉ giữ 1 đại diện cho mỗi `dup_group`** (loại ảnh hưởng của các chuỗi như *Blog chứng khoán*).

In [ ]:
tr_df = df.loc[TR].reset_index(drop=True)
keep = ~tr_df['dup_group'].duplicated().values
Xk, yk = Xcnt[keep], y_tr[keep]
rows = []
for c in CATS:
    a = set(vocab[np.argsort(log_odds_z(Xcnt, y_tr == c))[::-1][:10]])
    b = set(vocab[np.argsort(log_odds_z(Xk, yk == c))[::-1][:10]])
    rows.append({'Chuyên mục': c, 'Số bài sau gộp nhóm': int((yk == c).sum()), 'Số từ trùng top-10': len(a & b),
                 'Từ rơi khỏi top-10': ', '.join(sorted(a - b)) or '-'})
print(f"Train: {len(tr_df)} bài → {int(keep.sum())} đại diện nhóm")
display(pd.DataFrame(rows))

**Nhận xét:** @@sec9_5@@

## 10. Cụm từ (N-gram) theo độ phủ bài viết (Train)

Đếm **số bài chứa cụm** (document frequency) thay vì tổng số lần xuất hiện, để một bài lặp nhiều lần không lấn át. Bỏ cụm có stopword ở biên và cụm chứa chữ số.

In [ ]:
def extract_clean_ngrams(tokens_series, n=2, top_k=10):
    doc_freq = Counter()
    for tokens in tokens_series:
        seen = set()
        for i in range(len(tokens) - n + 1):
            w = tokens[i:i + n]
            if w[0] in STOPWORDS or w[-1] in STOPWORDS:
                continue
            if any(ch.isdigit() for t in w for ch in t):
                continue
            seen.add(' '.join(t.replace('_', ' ') for t in w))
        doc_freq.update(seen)
    return doc_freq.most_common(top_k)

tr_docs = df.loc[TR, 'seg_doc']; n_tr = len(tr_docs)
bi, tri = extract_clean_ngrams(tr_docs, 2, 10), extract_clean_ngrams(tr_docs, 3, 10)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.4))
for ax, data, col, ttl in [(axes[0], bi, '#457B9D', 'Top 10 bigram'), (axes[1], tri, '#E76F51', 'Top 10 trigram')]:
    w, c = zip(*data)
    ax.barh(list(w)[::-1], list(c)[::-1], color=col, edgecolor='#333', linewidth=.5)
    for i, v in enumerate(list(c)[::-1]):
        ax.text(v + max(c) * .01, i, f"{v} bài ({v / n_tr * 100:.1f}%)", va='center', fontsize=8.5)
    ax.set_xlim(0, max(c) * 1.3); ax.set_title(f"{ttl} theo độ phủ bài viết (Train)"); ax.set_xlabel("Số bài chứa cụm")
plt.show()

per_cat = pd.DataFrame({c: [f"{w} ({n})" for w, n in extract_clean_ngrams(df.loc[TR & (df['category'] == c).values, 'seg_doc'], 2, 6)] for c in CATS})
print("Top 6 bigram theo từng chuyên mục (số bài chứa cụm):")
display(per_cat)

**Nhận xét:** @@sec10@@

## 11. Thực thể số và ký hiệu tài chính

Mật độ phần trăm, số tiền và mã cổ phiếu là các đặc trưng "kiểu báo chí tài chính" có thể bổ sung cho TF-IDF. Mã cổ phiếu được đếm theo hai cách: **chặt** (đứng sau từ dẫn như *cổ phiếu, mã*) và **rộng** (cụm 3 chữ in hoa, loại trừ danh sách viết tắt không phải mã cổ phiếu — chỉ là ước lượng).

In [ ]:
PERCENT_RE = re.compile(r"\d+(?:[.,]\d+)?\s?%")
MONEY_RE = re.compile(r"\d[\d.,]*\s*(?:nghìn\s+tỷ|nghìn|ngàn|tỷ|triệu)?\s*(?:đồng|vnd|usd|đô la|eur|euro|nhân dân tệ|yen|đô)\b", re.IGNORECASE)
TICKER_STRICT_RE = re.compile(r"(?i:\bcổ phiếu|\bmã cổ phiếu|\bmã chứng khoán|\bmã)\s+([A-Z][A-Z0-9]{2,3})\b")
UPPER3_RE = re.compile(r"\b[A-Z]{3}\b")
NON_TICKER = {'USD', 'VND', 'GDP', 'FDI', 'CEO', 'CFO', 'ETF', 'ESG', 'SME', 'IPO', 'LNG', 'SJC', 'HCM', 'III', 'BOJ', 'WTI', 'PCE',
              'UAE', 'EUR', 'CPI', 'FED', 'VNI', 'IMF', 'WTO', 'EVN', 'HNX', 'OTC', 'NHNN', 'GDP', 'PMI', 'OPEC', 'ECB', 'AFP', 'CNY',
              'JPY', 'BRICS', 'FTA', 'WHO', 'NATO', 'EVFTA', 'AI', 'APEC', 'VAT', 'TPP', 'CPTPP', 'ASEAN', 'EU', 'US', 'UBCK', 'SGK'}
df['n_percent'] = df['content'].map(lambda x: len(PERCENT_RE.findall(x)))
df['n_money'] = df['content'].map(lambda x: len(MONEY_RE.findall(x)))
df['n_ticker_strict'] = df['content'].map(lambda x: len(TICKER_STRICT_RE.findall(x)))
df['n_ticker_loose'] = df['content'].map(lambda x: sum(t not in NON_TICKER for t in UPPER3_RE.findall(x)))

ent = df.groupby('category')[['n_percent', 'n_money', 'n_ticker_strict', 'n_ticker_loose']].mean().reindex(CATS).round(2)
ent.columns = ['% / bài', 'Tiền tệ / bài', 'Mã CP (chặt) / bài', 'Mã CP (rộng) / bài']
display(ent)
ent[['% / bài', 'Tiền tệ / bài', 'Mã CP (rộng) / bài']].plot(kind='bar', figsize=(9, 3.8), colormap='viridis', edgecolor='#333', linewidth=.5)
plt.title("Mật độ thực thể số theo chuyên mục"); plt.xticks(rotation=20, ha='right'); plt.ylabel("Số lần / bài"); plt.show()

**Nhận xét:** @@sec11@@

## 12. Kiểm toán thuộc tính tác giả (Author Leakage Audit)

Nếu phóng viên chuyên trách một chuyên mục thì tên tác giả (hoặc chữ ký cuối bài) là "đường tắt" để đoán nhãn mà không cần hiểu nội dung. Nhóm đo mức liên hệ bằng AMI/NMI, bằng một **bộ đoán nhãn theo tác giả** (leave-one-out), và kiểm tra tên tác giả có còn lọt vào thân bài không.

In [ ]:
def clean_author_name(a):
    a = "" if pd.isna(a) else str(a)
    a = re.sub(r"\(.*?\)", "", a)
    a = re.sub(r"\s+", " ", a).strip()
    return a if a else "Không rõ"

df['clean_author'] = df['author'].map(clean_author_name)
print(f"Số tác giả: {df['clean_author'].nunique()} | số bài của tác giả đông nhất: {df['clean_author'].value_counts().iloc[0]}")
print(f"AMI(tác giả, chuyên mục) = {adjusted_mutual_info_score(df['clean_author'], df['category']):.3f} | "
      f"NMI = {normalized_mutual_info_score(df['clean_author'], df['category']):.3f}")

# Bộ đoán nhãn LOO: nhãn = chuyên mục chiếm đa số trong các bài KHÁC của cùng tác giả
ct = pd.crosstab(df['clean_author'], df['category'])[CATS]
hits = tot = 0
for a, c in zip(df['clean_author'], df['category']):
    row = ct.loc[a].copy(); row[c] -= 1
    if row.sum() > 0:
        tot += 1; hits += (row.idxmax() == c)
print(f"Đoán chuyên mục chỉ từ tên tác giả (LOO): đúng {hits / tot * 100:.1f}% trên {tot:,} bài ({tot / len(df) * 100:.0f}% số bài có tác giả khác cùng viết) — mức ngẫu nhiên = 20%")

big = ct[ct.sum(axis=1) >= 5]
print(f"Trong {len(big)} tác giả có >= 5 bài: {(big.max(axis=1) / big.sum(axis=1) >= 0.9).mean() * 100:.0f}% có >= 90% bài thuộc một chuyên mục")

tail_hit = np.mean([a.lower() in c[-250:].lower() for a, c in zip(df['clean_author'], df['content']) if a != "Không rõ"])
body_hit = np.mean([a.lower() in c.lower() for a, c in zip(df['clean_author'], df['content']) if a != "Không rõ"])
print(f"Tên tác giả xuất hiện trong 250 ký tự cuối thân bài: {tail_hit * 100:.1f}% | ở bất kỳ vị trí nào trong thân bài: {body_hit * 100:.1f}%")

**Nhận xét:** @@sec12@@

## 13. Độ tách biệt giữa các chuyên mục và độ nhiễu nhãn

### 13.1 Tương đồng giữa tâm các chuyên mục (Train)

In [ ]:
tfidf = TfidfVectorizer(token_pattern=r"\S+", lowercase=False, stop_words=sorted(STOPWORDS), min_df=5, max_df=0.8, sublinear_tf=True, max_features=8000)
Xtf = tfidf.fit_transform(df.loc[TR, 'tok_doc_str']).tocsr()
centroids = Normalizer().fit_transform(np.vstack([np.asarray(Xtf[y_tr == c].mean(axis=0)).ravel() for c in CATS]))
sim_df = pd.DataFrame(centroids @ centroids.T, index=CATS, columns=CATS)

plt.figure(figsize=(6.5, 4.6))
sns.heatmap(sim_df, annot=True, fmt='.3f', cmap='YlGnBu', vmin=0.2, vmax=1.0)
plt.title("Cosine giữa các tâm chuyên mục (Train)"); plt.show()

off = sim_df.where(~np.eye(5, dtype=bool)).stack().sort_values(ascending=False)
pairs_sorted = [(a, b, round(v, 3)) for (a, b), v in off.items() if CATS.index(a) < CATS.index(b)]
print("Cặp gần nhau nhất :", pairs_sorted[:3])
print("Cặp xa nhau nhất  :", pairs_sorted[-3:])
print("Tâm 'Thế giới' có cosine trung bình với 4 lớp còn lại =", round(sim_df.loc['Thế giới'].drop('Thế giới').mean(), 3),
      "| các lớp khác:", {c: round(sim_df.loc[c].drop(c).mean(), 3) for c in CATS if c != 'Thế giới'})

**Nhận xét:** @@sec13_1@@

### 13.2 Baseline có giám sát và kiểm toán nhãn đáng ngờ (Train+Val)

Để **định lượng** độ nhập nhằng nhãn (thay vì suy đoán từ ma trận tâm), nhóm chạy một *probe* đơn giản: TF-IDF (1–2-gram) + Logistic Regression, đánh giá bằng 5-fold `StratifiedGroupKFold` trên Train+Val (nhóm = `dup_group`, vector hóa được fit **bên trong từng fold**). Đây không phải mô hình cuối cùng (chưa tinh chỉnh siêu tham số) mà chỉ để đo độ tách biệt lớp và tìm nhãn đáng ngờ. Tập Test không được dùng.

In [ ]:
dev = df[DEV].reset_index(drop=True)
X_dev, y_dev, g_dev = dev['tok_doc_str'], dev['category'].values, dev['dup_group'].values

probe = Pipeline([
    ('tfidf', TfidfVectorizer(token_pattern=r"\S+", lowercase=False, stop_words=sorted(STOPWORDS),
                              ngram_range=(1, 2), min_df=3, max_df=0.9, sublinear_tf=True)),
    ('clf', LogisticRegression(C=10, max_iter=2000))])
sgkf5 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
proba = cross_val_predict(probe, X_dev, y_dev, groups=g_dev, cv=sgkf5, method='predict_proba', n_jobs=1)
classes = np.array(sorted(set(y_dev)))
y_pred = classes[proba.argmax(1)]

print(f"Accuracy (CV) = {(y_pred == y_dev).mean():.4f} | Macro-F1 (CV) = {f1_score(y_dev, y_pred, average='macro'):.4f}")
display(pd.DataFrame(classification_report(y_dev, y_pred, output_dict=True, zero_division=0)).T.round(3).loc[CATS + ['macro avg']])

cm = confusion_matrix(y_dev, y_pred, labels=CATS)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
plt.figure(figsize=(6.8, 5))
sns.heatmap(pd.DataFrame(cm_norm, index=CATS, columns=CATS), annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1)
plt.title("Ma trận nhầm lẫn chuẩn hóa theo hàng (CV, Train+Val)"); plt.xlabel("Dự đoán"); plt.ylabel("Nhãn thật"); plt.show()

conf = [(CATS[i], CATS[j], int(cm[i, j])) for i in range(5) for j in range(5) if i != j]
print("Các nhầm lẫn nhiều nhất (thật → dự đoán, số bài):", sorted(conf, key=lambda t: -t[2])[:5])

**Nhận xét:** @@sec13_2@@

**Danh sách bài có nhãn đáng ngờ:** bài mà mô hình (không nhìn thấy bài đó khi huấn luyện) dự đoán một lớp *khác* nhãn với độ tin cậy cao. Đây là ứng viên cần đọc thủ công.

In [ ]:
p_pred = proba.max(1)
p_true = proba[np.arange(len(y_dev)), [list(classes).index(c) for c in y_dev]]
sus = dev.assign(pred=y_pred, p_pred=p_pred.round(3), p_true=p_true.round(3))
sus = sus[sus['pred'] != sus['category']]
n_hi = int((sus['p_pred'] >= 0.8).sum())
print(f"Dự đoán khác nhãn: {len(sus)} bài ({len(sus) / len(dev) * 100:.1f}% Train+Val); trong đó tin cậy >= 0.8: {n_hi} bài ({n_hi / len(dev) * 100:.1f}%)")
display(sus.sort_values('p_pred', ascending=False)[['title', 'category', 'pred', 'p_pred']].head(12).rename(columns={'category': 'nhãn gốc', 'pred': 'dự đoán'}))

**Nhận xét:** @@sec13_3@@

## 14. Tổng kết EDA

@@summary@@